# Design a Machine Learning Approach to Analyse Students’ Performance Based on Their Socio-economic Status in the Kingdom of Bahrain

**MSc Artificial Intelligence — Bahrain Polytechnic**  
**Student: Jaafar Ahmed | ID: 202508989**

# Phase 8 — Final Model, Locked Holdout Evaluation, and Prediction Pipeline

Phase 8 locks the selected model **before** the final holdout data are opened, verifies the complete data lineage from Phase 1 to Phase 7, evaluates the locked model once on the untouched holdout set, and exports a final prediction bundle and model card.

## Important methodological safeguard

The current Phase 4 hyperparameter search is scheduled for deeper future tuning. Therefore, this notebook starts in **`DRY_RUN` mode** and keeps the holdout sealed. The final holdout must be opened only after the Phase 4 tuning design and all model-selection decisions are frozen.

```text
Phase 6 primary candidate + Phase 7 interpretation
                    ↓
Lock model identity and artifact before holdout access
                    ↓
Refit/verify on the complete Phase 3 training set
                    ↓
DRY_RUN: holdout remains sealed
        OR
FINAL_HOLDOUT: open the untouched holdout once
                    ↓
Final metrics, uncertainty, subgroup audit, model card
                    ↓
Saved prediction pipeline and final research outputs
```

# Section 8.1 — Configuration and Final-Evaluation Governance

**Purpose:** Configure the project, define the holdout lock, and create portable output folders without reading the holdout dataset.

In [1]:
# Cell 8.1.1 — Configure Environment and Final Holdout Lock

from pathlib import Path
from datetime import datetime
import hashlib
import importlib.util
import json
import math
import random
import re
import shutil
import subprocess
import sys
import time
import warnings
import zipfile

# -----------------------------------------------------------------------------
# CHANGE THESE SETTINGS ONLY
# -----------------------------------------------------------------------------
EXECUTION_ENVIRONMENT = "COLAB"  # "COLAB", "ANACONDA", or "VSCODE"

# Keep DRY_RUN until the expanded Phase 4 tuning is final and frozen.
PHASE_8_MODE = "DRY_RUN"  # "DRY_RUN" or "FINAL_HOLDOUT"
FINAL_HOLDOUT_UNLOCK_PHRASE = ""
REQUIRED_UNLOCK_PHRASE = "OPEN THE FINAL HOLDOUT ONCE"

REFIT_FINAL_MODEL_ON_FULL_TRAINING = True
PROVISIONAL_UNTIL_PHASE_4_RETUNED = True
BOOTSTRAP_ITERATIONS = 1000
CALIBRATION_BINS = 10
LOW_CONFIDENCE_THRESHOLD = 0.55
MINIMUM_SUBGROUP_SIZE = 20
RANDOM_STATE = 42
AUTO_INSTALL_MISSING_PACKAGES = True

PHASE_7_MANIFEST_OVERRIDE = None
PHASE_3_MANIFEST_OVERRIDE = None
TRAINING_DATA_OVERRIDE = None
HOLDOUT_DATA_OVERRIDE = None
FINAL_MODEL_ARTIFACT_OVERRIDE = None

# Optional portable output overrides for testing only.
PHASE_OUTPUT_OVERRIDE = None
OFFICIAL_OUTPUT_DATA_DIR_OVERRIDE = None
GOVERNANCE_DIR_OVERRIDE = None
# -----------------------------------------------------------------------------

PROJECT_TITLE = (
    "Design a Machine Learning Approach to Analyse Students’ Performance "
    "Based on Their Socio-economic Status in the Kingdom of Bahrain"
)
PROGRAMME = "MSc Artificial Intelligence — Bahrain Polytechnic"
STUDENT_NAME = "Jaafar Ahmed"
STUDENT_ID = "202508989"

random.seed(RANDOM_STATE)
PHASE_8_MODE = PHASE_8_MODE.strip().upper()
if PHASE_8_MODE not in {"DRY_RUN", "FINAL_HOLDOUT"}:
    raise ValueError("PHASE_8_MODE must be DRY_RUN or FINAL_HOLDOUT.")

if EXECUTION_ENVIRONMENT.strip().upper() == "COLAB":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_ROOT = Path("/content/drive/MyDrive/Jaafar_MSc_Thesis")
    except ImportError:
        PROJECT_ROOT = Path.cwd()
        print("Google Colab is unavailable; using:", PROJECT_ROOT)
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
PHASE_DIR = (
    Path(PHASE_OUTPUT_OVERRIDE)
    if PHASE_OUTPUT_OVERRIDE
    else OUTPUT_ROOT / "Phase_08_Final_Model_and_Holdout_Evaluation"
)
OFFICIAL_DATA_DIR = (
    Path(OFFICIAL_OUTPUT_DATA_DIR_OVERRIDE)
    if OFFICIAL_OUTPUT_DATA_DIR_OVERRIDE
    else DATA_DIR
)
MODEL_DIR = PHASE_DIR / "models"
GOVERNANCE_DIR = (
    Path(GOVERNANCE_DIR_OVERRIDE)
    if GOVERNANCE_DIR_OVERRIDE
    else PROJECT_ROOT / "governance"
)

for folder in [DATA_DIR, OUTPUT_ROOT, PHASE_DIR, OFFICIAL_DATA_DIR, MODEL_DIR, GOVERNANCE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
RUN_ID = f"phase8_{RUN_TIMESTAMP}"
FINAL_MODE = PHASE_8_MODE == "FINAL_HOLDOUT"

print("Project root:", PROJECT_ROOT)
print("Phase 8 mode:", PHASE_8_MODE)
print("Final holdout unlocked:", FINAL_MODE)

Google Colab is unavailable; using: C:\Users\User\Desktop\All
Project root: C:\Users\User\Desktop\All
Phase 8 mode: DRY_RUN
Final holdout unlocked: False


In [2]:
# Cell 8.1.2 — Install Libraries, Import Packages, and Define Core Utilities

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
}

for import_name, package_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        if not AUTO_INSTALL_MISSING_PACKAGES:
            raise ImportError(f"Missing package: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("default")
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

THEME = {
    "maroon": "#6B1E32",
    "deep_maroon": "#43101F",
    "gold": "#D4A64A",
    "teal": "#237A7A",
    "coral": "#D96850",
    "indigo": "#525B92",
    "sage": "#7A9B76",
    "sand": "#D9C7A3",
    "warm_white": "#F7F4EF",
    "charcoal": "#2E3035",
    "soft_grey": "#D8D5D0",
}

CLASS_COLORS = {
    "Low": THEME["coral"],
    "Medium": THEME["gold"],
    "High": THEME["teal"],
}

mpl.rcParams.update({
    "figure.facecolor": THEME["warm_white"],
    "axes.facecolor": THEME["warm_white"],
    "savefig.facecolor": THEME["warm_white"],
    "text.color": THEME["charcoal"],
    "axes.labelcolor": THEME["charcoal"],
    "xtick.color": THEME["charcoal"],
    "ytick.color": THEME["charcoal"],
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "font.size": 10,
    "legend.frameon": False,
})


def safe_name(text):
    text = str(text).strip().replace("—", "-").replace("–", "-")
    text = re.sub(r"[^\w\-]+", "_", text, flags=re.UNICODE)
    return re.sub(r"_+", "_", text).strip("_") or "Unnamed"


def create_cell_folders(section_number, section_title, cell_number, cell_title):
    base = (
        PHASE_DIR
        / safe_name(f"Section_{section_number}_{section_title}")
        / safe_name(f"Cell_{cell_number}_{cell_title}")
    )
    folders = {
        "tables": base / "tables",
        "figures": base / "figures",
        "reports": base / "reports",
        "models": base / "models",
        "files": base / "files",
    }
    for folder in folders.values():
        folder.mkdir(parents=True, exist_ok=True)
    return folders


def save_dataframe(dataframe, folders, file_stem, index=False):
    path = folders["tables"] / f"{file_stem}.csv"
    dataframe.to_csv(path, index=index, encoding="utf-8-sig")
    return path


def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(data, handle, ensure_ascii=False, indent=2, default=str)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_table(path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, encoding="utf-8-sig")
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported data format: {path.suffix}")


def resolve_artifact(override, recorded_path, known_relative_paths, description):
    candidates = []
    if override:
        candidates.append(Path(override).expanduser())
    if recorded_path:
        candidates.append(Path(recorded_path).expanduser())
    candidates.extend(PROJECT_ROOT / Path(relative) for relative in known_relative_paths)

    for candidate in candidates:
        if candidate.exists():
            return candidate

    basenames = [candidate.name for candidate in candidates if candidate.name]
    matches = []
    for basename in dict.fromkeys(basenames):
        matches.extend(PROJECT_ROOT.rglob(basename))
    matches = [path for path in matches if path.is_file()]

    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        preferred = [path for path in matches if "outputs" in path.parts or "data" in path.parts]
        if len(preferred) == 1:
            return preferred[0]
        raise FileNotFoundError(
            f"Multiple files matched {description}: " + ", ".join(str(path) for path in matches[:10])
        )
    raise FileNotFoundError(f"Could not locate {description}.")


def style_axis(axis, title, subtitle=None, grid_axis="y"):
    axis.set_title(title, loc="left", pad=18, fontweight="bold")
    if subtitle:
        axis.text(0, 1.015, subtitle, transform=axis.transAxes, fontsize=9, color="#5B5B5B")
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.spines["left"].set_alpha(0.35)
    axis.spines["bottom"].set_alpha(0.35)
    if grid_axis:
        axis.grid(axis=grid_axis, alpha=0.16)
    axis.set_axisbelow(True)


def save_figure(figure, folders, file_stem):
    path = folders["figures"] / f"{file_stem}.png"
    figure.text(
        0.01,
        0.01,
        "MSc Artificial Intelligence — Bahrain Polytechnic | Phase 8",
        fontsize=7.5,
        color="#666666",
    )
    figure.savefig(path, dpi=300, bbox_inches="tight")
    return path

# Section 8.2 — Lock the Final Candidate before Holdout Access

**Purpose:** Read Phase 7 and Phase 3 manifests, select the Phase 6 primary candidate, record its identity and hashes, and freeze this decision before any holdout file is loaded.

In [3]:
# Cell 8.2.1 — Locate Manifests and Lock the Primary Candidate

CELL_FOLDERS = create_cell_folders(
    "08_02", "Lock_Final_Candidate", "08_02_01", "Locate_Manifests_and_Lock_Primary_Candidate"
)

PHASE_7_MANIFEST_PATH = resolve_artifact(
    PHASE_7_MANIFEST_OVERRIDE,
    None,
    [
        "phase_07_handoff_manifest_for_phase_08.json",
        "outputs/Phase_07_Model_Explainability_and_Interpretation/phase_07_handoff_manifest_for_phase_08.json",
    ],
    "Phase 7 handoff manifest",
)

PHASE_3_MANIFEST_PATH = resolve_artifact(
    PHASE_3_MANIFEST_OVERRIDE,
    None,
    [
        "phase_03_handoff_manifest_for_phase_04.json",
        "outputs/Phase_03_Train_Test_Split_and_Preprocessing_Pipelines/phase_03_handoff_manifest_for_phase_04.json",
    ],
    "Phase 3 handoff manifest",
)

with PHASE_7_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    phase_7_manifest = json.load(handle)
with PHASE_3_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    phase_3_manifest = json.load(handle)

primary_candidate = phase_7_manifest["phase_6_primary_candidate"].copy()
recorded_model_path = primary_candidate.get("model_path")
model_filename = Path(recorded_model_path).name

FINAL_MODEL_SOURCE_PATH = resolve_artifact(
    FINAL_MODEL_ARTIFACT_OVERRIDE,
    recorded_model_path,
    [
        f"outputs/Phase_06_Model_Comparison_and_Quality_Checking/selected_candidate_models/{model_filename}",
        model_filename,
    ],
    "locked final candidate model",
)

TRAINING_DATA_PATH = resolve_artifact(
    TRAINING_DATA_OVERRIDE,
    phase_7_manifest.get("training_dataset"),
    ["data/phase_03_training_dataset_for_phase_04.csv"],
    "Phase 3 training dataset",
)

candidate_lock = {
    "run_id": RUN_ID,
    "locked_before_holdout_access": True,
    "model": primary_candidate["model"],
    "stage": primary_candidate["stage"],
    "source_model_path": str(FINAL_MODEL_SOURCE_PATH),
    "source_model_sha256": sha256_file(FINAL_MODEL_SOURCE_PATH),
    "training_dataset_path": str(TRAINING_DATA_PATH),
    "training_dataset_sha256": sha256_file(TRAINING_DATA_PATH),
    "phase_7_manifest_path": str(PHASE_7_MANIFEST_PATH),
    "phase_7_manifest_sha256": sha256_file(PHASE_7_MANIFEST_PATH),
    "phase_3_manifest_path": str(PHASE_3_MANIFEST_PATH),
    "phase_3_manifest_sha256": sha256_file(PHASE_3_MANIFEST_PATH),
    "phase_8_mode": PHASE_8_MODE,
    "provisional_until_phase_4_retuned": PROVISIONAL_UNTIL_PHASE_4_RETUNED,
    "locked_at": datetime.now().isoformat(),
}

CANDIDATE_LOCK_PATH = save_json(
    candidate_lock,
    GOVERNANCE_DIR / "phase_08_locked_candidate.json",
)
save_json(candidate_lock, CELL_FOLDERS["reports"] / "08_02_01_locked_candidate.json")

print("Locked model:", candidate_lock["model"], "—", candidate_lock["stage"])
print("Candidate locked before holdout access:", True)
print("Holdout loaded:", False)

Locked model: Logistic Regression — Baseline
Candidate locked before holdout access: True
Holdout loaded: False


# Section 8.3 — Verify Training Lineage and Build the Final Full-Training Model

**Purpose:** Verify the exact Phase 3 training data, load the locked candidate, optionally refit it on the full training set, and save a portable prediction bundle.

In [4]:
# Cell 8.3.1 — Verify Training Dataset Integrity and Variable Roles

CELL_FOLDERS = create_cell_folders(
    "08_03", "Training_Lineage_and_Final_Model", "08_03_01", "Verify_Training_Dataset_Integrity"
)

training_data = load_table(TRAINING_DATA_PATH)

ID_COLUMN = phase_3_manifest["identifier_column"]
GROUP_COLUMN = phase_3_manifest["group_column"]
TARGET_COLUMN = phase_3_manifest["target_column"]
PREDICTOR_COLUMNS = phase_3_manifest["predictor_columns"]
TARGET_MAPPING = phase_7_manifest.get("target_mapping", {"Low": 0, "Medium": 1, "High": 2})
CLASS_ORDER = [name for name, code_value in sorted(TARGET_MAPPING.items(), key=lambda item: item[1])]
INVERSE_TARGET_MAPPING = {value: key for key, value in TARGET_MAPPING.items()}

required_columns = [ID_COLUMN, GROUP_COLUMN, *PREDICTOR_COLUMNS, TARGET_COLUMN]
missing_columns = sorted(set(required_columns) - set(training_data.columns))
if missing_columns:
    raise ValueError("Training data are missing required columns: " + ", ".join(missing_columns))

actual_training_hash = sha256_file(TRAINING_DATA_PATH)
expected_training_hash = phase_3_manifest.get("training_dataset_sha256")
if expected_training_hash and actual_training_hash != expected_training_hash:
    raise ValueError("Training dataset SHA-256 does not match the Phase 3 manifest.")
if len(training_data) != int(phase_3_manifest["training_records"]):
    raise ValueError("Training record count does not match the Phase 3 manifest.")

X_train = training_data[PREDICTOR_COLUMNS].copy()
y_train_names = training_data[TARGET_COLUMN].astype(str).copy()
y_train_encoded = y_train_names.map(TARGET_MAPPING).astype(int)
training_groups = training_data[GROUP_COLUMN].copy()

lineage_summary = pd.DataFrame([
    ["Training dataset", str(TRAINING_DATA_PATH)],
    ["Training records", len(training_data)],
    ["Training families", training_groups.nunique()],
    ["Predictors", len(PREDICTOR_COLUMNS)],
    ["Training SHA-256", actual_training_hash],
    ["Expected SHA-256 matched", True],
    ["Target classes", ", ".join(CLASS_ORDER)],
    ["Holdout loaded", False],
])
lineage_summary.columns = ["lineage_measure", "value"]
save_dataframe(lineage_summary, CELL_FOLDERS, "08_03_01_training_lineage_summary")
display(lineage_summary)

,lineage_measure,value
0,Training dataset,C:\Users\User\Desktop\All\data\phase_03_traini...
1,Training records,2006
2,Training families,984
3,Predictors,17
4,Training SHA-256,0efdd6e3754f8844d7eba3a92b70a27f8e5d202c660ea9...
5,Expected SHA-256 matched,True
6,Target classes,"Low, Medium, High"
7,Holdout loaded,False


In [5]:
# Cell 8.3.2 — Refit or Verify the Locked Candidate on Full Training Data

CELL_FOLDERS = create_cell_folders(
    "08_03", "Training_Lineage_and_Final_Model", "08_03_02", "Refit_or_Verify_Locked_Candidate"
)

source_artifact = joblib.load(FINAL_MODEL_SOURCE_PATH)


def refit_artifact(artifact, X, y_encoded):
    """Refit sklearn pipelines or the Phase 4 CatBoost bundle."""
    if isinstance(artifact, dict) and {"preprocessor", "model"}.issubset(artifact):
        preprocessor = clone(artifact["preprocessor"])
        transformed = preprocessor.fit_transform(X[PREDICTOR_COLUMNS])
        source_model = artifact["model"]
        model_class = source_model.__class__
        model_params = source_model.get_params()
        model = model_class(**model_params)
        model.fit(transformed, y_encoded)
        return {
            "preprocessor": preprocessor,
            "model": model,
            "target_mapping": TARGET_MAPPING,
            "predictors": PREDICTOR_COLUMNS,
        }

    fitted_model = clone(artifact)
    fitted_model.fit(X[PREDICTOR_COLUMNS], y_encoded)
    return fitted_model


start_time = time.perf_counter()
if REFIT_FINAL_MODEL_ON_FULL_TRAINING:
    fitted_final_artifact = refit_artifact(source_artifact, X_train, y_train_encoded)
    fit_action = "Refitted on the complete Phase 3 training set"
else:
    fitted_final_artifact = source_artifact
    fit_action = "Verified existing full-training artifact"
fit_seconds = time.perf_counter() - start_time

FINAL_MODEL_BUNDLE = {
    "artifact": fitted_final_artifact,
    "model_name": primary_candidate["model"],
    "model_stage": primary_candidate["stage"],
    "predictor_columns": PREDICTOR_COLUMNS,
    "target_mapping": TARGET_MAPPING,
    "class_order": CLASS_ORDER,
    "identifier_column": ID_COLUMN,
    "group_column": GROUP_COLUMN,
    "training_dataset_sha256": actual_training_hash,
    "candidate_lock": candidate_lock,
    "refit_action": fit_action,
    "refit_seconds": fit_seconds,
    "created_at": datetime.now().isoformat(),
}

FINAL_MODEL_BUNDLE_PATH = MODEL_DIR / "phase_08_final_model_bundle.joblib"
joblib.dump(FINAL_MODEL_BUNDLE, FINAL_MODEL_BUNDLE_PATH)


def raw_model_components(bundle):
    artifact = bundle["artifact"]
    if isinstance(artifact, dict) and {"preprocessor", "model"}.issubset(artifact):
        return artifact["preprocessor"], artifact["model"]
    return None, artifact


def predict_encoded(bundle, X):
    artifact = bundle["artifact"]
    predictors = bundle["predictor_columns"]
    if isinstance(artifact, dict) and {"preprocessor", "model"}.issubset(artifact):
        transformed = artifact["preprocessor"].transform(X[predictors])
        predictions = artifact["model"].predict(transformed)
    else:
        predictions = artifact.predict(X[predictors])
    return np.asarray(predictions).reshape(-1).astype(int)


def probability_scores(bundle, X):
    artifact = bundle["artifact"]
    predictors = bundle["predictor_columns"]
    if isinstance(artifact, dict) and {"preprocessor", "model"}.issubset(artifact):
        transformed = artifact["preprocessor"].transform(X[predictors])
        model = artifact["model"]
        input_data = transformed
    else:
        model = artifact
        input_data = X[predictors]

    if hasattr(model, "predict_proba"):
        probabilities = np.asarray(model.predict_proba(input_data), dtype=float)
    elif hasattr(model, "decision_function"):
        decision = np.asarray(model.decision_function(input_data), dtype=float)
        if decision.ndim == 1:
            decision = np.column_stack([-decision, decision])
        probabilities = softmax(decision, axis=1)
    else:
        encoded_predictions = predict_encoded(bundle, X)
        probabilities = np.zeros((len(encoded_predictions), len(CLASS_ORDER)), dtype=float)
        probabilities[np.arange(len(encoded_predictions)), encoded_predictions] = 1.0

    probabilities = np.clip(probabilities, 1e-12, None)
    probabilities = probabilities / probabilities.sum(axis=1, keepdims=True)

    classes = getattr(model, "classes_", np.arange(probabilities.shape[1]))
    classes = np.asarray(classes).reshape(-1)
    desired_codes = np.array([TARGET_MAPPING[name] for name in CLASS_ORDER])
    if len(classes) == probabilities.shape[1] and set(classes.tolist()) == set(desired_codes.tolist()):
        reorder = [int(np.where(classes == code_value)[0][0]) for code_value in desired_codes]
        probabilities = probabilities[:, reorder]
    return probabilities


smoke_predictions = predict_encoded(FINAL_MODEL_BUNDLE, X_train.head(10))
smoke_probabilities = probability_scores(FINAL_MODEL_BUNDLE, X_train.head(10))
if smoke_probabilities.shape != (10, len(CLASS_ORDER)):
    raise ValueError("Final model probability output has an unexpected shape.")
if not np.allclose(smoke_probabilities.sum(axis=1), 1.0, atol=1e-6):
    raise ValueError("Final model probabilities do not sum to one.")

model_build_summary = pd.DataFrame([
    ["Locked model", primary_candidate["model"]],
    ["Locked stage", primary_candidate["stage"]],
    ["Full-training action", fit_action],
    ["Training seconds", fit_seconds],
    ["Smoke-test records", 10],
    ["Prediction classes", ", ".join(CLASS_ORDER)],
    ["Final model bundle", str(FINAL_MODEL_BUNDLE_PATH)],
    ["Holdout loaded", False],
])
model_build_summary.columns = ["model_build_measure", "value"]
save_dataframe(model_build_summary, CELL_FOLDERS, "08_03_02_final_model_build_summary")
display(model_build_summary)

,model_build_measure,value
0,Locked model,Logistic Regression
1,Locked stage,Baseline
2,Full-training action,Refitted on the complete Phase 3 training set
3,Training seconds,0.292793
4,Smoke-test records,10
5,Prediction classes,"Low, Medium, High"
6,Final model bundle,C:\Users\User\Desktop\All\outputs\Phase_08_Fin...
7,Holdout loaded,False


# Section 8.4 — Final Holdout Gate

**Purpose:** Keep the holdout sealed during Dry Run and require an explicit unlock phrase before the final evaluation. The locked candidate cannot change after this point.

In [6]:
# Cell 8.4.1 — Enforce the Final Holdout Gate and One-Time Access Log

CELL_FOLDERS = create_cell_folders(
    "08_04", "Final_Holdout_Gate", "08_04_01", "Enforce_Final_Holdout_Gate"
)

HOLDOUT_ACCESS_LOG_PATH = GOVERNANCE_DIR / "phase_08_holdout_access_log.json"
holdout_data = None
HOLDOUT_DATA_PATH = None

if not FINAL_MODE:
    print("DRY_RUN completed: the untouched holdout remains sealed.")
    print("After Phase 4 tuning is final, set:")
    print('PHASE_8_MODE = "FINAL_HOLDOUT"')
    print(f'FINAL_HOLDOUT_UNLOCK_PHRASE = "{REQUIRED_UNLOCK_PHRASE}"')
else:
    if FINAL_HOLDOUT_UNLOCK_PHRASE != REQUIRED_UNLOCK_PHRASE:
        raise PermissionError("The final holdout unlock phrase is missing or incorrect.")
    if HOLDOUT_ACCESS_LOG_PATH.exists():
        previous_access = json.load(HOLDOUT_ACCESS_LOG_PATH.open("r", encoding="utf-8"))
        raise RuntimeError(
            "The final holdout access log already exists. Do not repeat model evaluation on the holdout. "
            f"Previous access: {previous_access.get('accessed_at')}"
        )

    HOLDOUT_DATA_PATH = resolve_artifact(
        HOLDOUT_DATA_OVERRIDE,
        phase_3_manifest.get("untouched_holdout_test_for_phase_8")
        or phase_7_manifest.get("holdout_test_path_recorded_but_not_loaded"),
        ["data/phase_03_untouched_holdout_test_for_phase_08.csv"],
        "untouched Phase 3 holdout dataset",
    )

    holdout_access_record = {
        "run_id": RUN_ID,
        "candidate_lock_path": str(CANDIDATE_LOCK_PATH),
        "candidate_lock_sha256": sha256_file(CANDIDATE_LOCK_PATH),
        "locked_model": primary_candidate["model"],
        "locked_stage": primary_candidate["stage"],
        "holdout_path": str(HOLDOUT_DATA_PATH),
        "accessed_at": datetime.now().isoformat(),
        "purpose": "One-time final Phase 8 evaluation only",
        "model_selection_on_holdout_prohibited": True,
    }
    save_json(holdout_access_record, HOLDOUT_ACCESS_LOG_PATH)
    save_json(holdout_access_record, CELL_FOLDERS["reports"] / "08_04_01_holdout_access_record.json")

    holdout_data = load_table(HOLDOUT_DATA_PATH)
    print("FINAL_HOLDOUT mode: untouched holdout loaded once.")
    print("Holdout shape:", holdout_data.shape)

DRY_RUN completed: the untouched holdout remains sealed.
After Phase 4 tuning is final, set:
PHASE_8_MODE = "FINAL_HOLDOUT"
FINAL_HOLDOUT_UNLOCK_PHRASE = "OPEN THE FINAL HOLDOUT ONCE"


# Section 8.5 — Holdout Integrity and Final Predictions

**Purpose:** Verify the holdout hash, record count, variables, and family separation, then generate the only final predictions from the previously locked model.

In [7]:
# Cell 8.5.1 — Verify Holdout Integrity and Family Separation

CELL_FOLDERS = create_cell_folders(
    "08_05", "Holdout_Integrity_and_Predictions", "08_05_01", "Verify_Holdout_Integrity"
)

holdout_integrity = pd.DataFrame()
X_holdout = None
y_holdout_names = None
y_holdout_encoded = None
holdout_groups = None

if FINAL_MODE:
    missing_holdout_columns = sorted(set(required_columns) - set(holdout_data.columns))
    if missing_holdout_columns:
        raise ValueError("Holdout data are missing required columns: " + ", ".join(missing_holdout_columns))

    actual_holdout_hash = sha256_file(HOLDOUT_DATA_PATH)
    expected_holdout_hash = phase_3_manifest.get("holdout_dataset_sha256")
    if expected_holdout_hash and actual_holdout_hash != expected_holdout_hash:
        raise ValueError("Holdout SHA-256 does not match the Phase 3 manifest.")
    if len(holdout_data) != int(phase_3_manifest["holdout_records"]):
        raise ValueError("Holdout record count does not match the Phase 3 manifest.")

    X_holdout = holdout_data[PREDICTOR_COLUMNS].copy()
    y_holdout_names = holdout_data[TARGET_COLUMN].astype(str).copy()
    y_holdout_encoded = y_holdout_names.map(TARGET_MAPPING).astype(int)
    holdout_groups = holdout_data[GROUP_COLUMN].copy()

    family_overlap = len(set(training_groups.unique()).intersection(set(holdout_groups.unique())))
    if family_overlap != 0:
        raise ValueError("Family leakage detected between training and holdout sets.")

    holdout_integrity = pd.DataFrame([
        ["Holdout dataset", str(HOLDOUT_DATA_PATH)],
        ["Holdout records", len(holdout_data)],
        ["Holdout families", holdout_groups.nunique()],
        ["Expected SHA-256 matched", True],
        ["Family overlap with training", family_overlap],
        ["Target classes present", ", ".join(sorted(y_holdout_names.unique()))],
        ["Locked model unchanged", candidate_lock["source_model_sha256"] == sha256_file(FINAL_MODEL_SOURCE_PATH)],
    ], columns=["integrity_measure", "value"])
    save_dataframe(holdout_integrity, CELL_FOLDERS, "08_05_01_holdout_integrity")
    display(holdout_integrity)
else:
    print("Skipped: final holdout remains sealed in DRY_RUN mode.")

Skipped: final holdout remains sealed in DRY_RUN mode.


In [8]:
# Cell 8.5.2 — Generate Final Holdout Predictions and Probabilities

CELL_FOLDERS = create_cell_folders(
    "08_05", "Holdout_Integrity_and_Predictions", "08_05_02", "Generate_Final_Holdout_Predictions"
)

final_predictions = pd.DataFrame()
final_prediction_seconds = np.nan

if FINAL_MODE:
    prediction_start = time.perf_counter()
    y_pred_encoded = predict_encoded(FINAL_MODEL_BUNDLE, X_holdout)
    y_probabilities = probability_scores(FINAL_MODEL_BUNDLE, X_holdout)
    final_prediction_seconds = time.perf_counter() - prediction_start

    y_pred_names = pd.Series(y_pred_encoded).map(INVERSE_TARGET_MAPPING).to_numpy()
    confidence = y_probabilities.max(axis=1)
    sorted_probabilities = np.sort(y_probabilities, axis=1)
    probability_margin = sorted_probabilities[:, -1] - sorted_probabilities[:, -2]
    entropy = -np.sum(y_probabilities * np.log(y_probabilities), axis=1)
    maximum_entropy = math.log(len(CLASS_ORDER))
    normalised_entropy = entropy / maximum_entropy

    true_codes = y_holdout_encoded.to_numpy()
    error_distance = np.abs(true_codes - y_pred_encoded)

    final_predictions = pd.DataFrame({
        ID_COLUMN: holdout_data[ID_COLUMN].values,
        GROUP_COLUMN: holdout_data[GROUP_COLUMN].values,
        "true_class": y_holdout_names.values,
        "predicted_class": y_pred_names,
        "correct_prediction": y_holdout_names.values == y_pred_names,
        "prediction_confidence": confidence,
        "probability_margin": probability_margin,
        "normalised_entropy": normalised_entropy,
        "error_distance": error_distance,
        "error_type": np.select(
            [error_distance == 0, error_distance == 1, error_distance >= 2],
            ["Correct", "Adjacent-class error", "Extreme Low–High error"],
            default="Unknown",
        ),
        "human_review_recommended": confidence < LOW_CONFIDENCE_THRESHOLD,
    })

    for class_index, class_name in enumerate(CLASS_ORDER):
        final_predictions[f"probability_{class_name.lower()}"] = y_probabilities[:, class_index]

    save_dataframe(final_predictions, CELL_FOLDERS, "08_05_02_final_holdout_predictions")
    display(final_predictions.head())
else:
    print("Skipped: no holdout predictions were generated in DRY_RUN mode.")

Skipped: no holdout predictions were generated in DRY_RUN mode.


# Section 8.6 — Comprehensive Final Metrics

**Purpose:** Calculate overall, macro, weighted, class-specific, probability, agreement, and error-safety metrics without changing the locked model or prediction rule.

In [9]:
# Cell 8.6.1 — Calculate Overall and Class-Specific Holdout Metrics

CELL_FOLDERS = create_cell_folders(
    "08_06", "Comprehensive_Final_Metrics", "08_06_01", "Calculate_Final_Holdout_Metrics"
)

final_metrics = pd.DataFrame()
per_class_metrics = pd.DataFrame()
classification_report_table = pd.DataFrame()
confusion_counts = pd.DataFrame()
confusion_normalised = pd.DataFrame()

if FINAL_MODE:
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_holdout_names, y_pred_names, labels=CLASS_ORDER, average="macro", zero_division=0
    )
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_holdout_names, y_pred_names, labels=CLASS_ORDER, average="weighted", zero_division=0
    )
    class_precision, class_recall, class_f1, class_support = precision_recall_fscore_support(
        y_holdout_names, y_pred_names, labels=CLASS_ORDER, average=None, zero_division=0
    )

    y_binary = label_binarize(y_holdout_names, classes=CLASS_ORDER)
    brier_multiclass = np.mean(np.sum((y_probabilities - y_binary) ** 2, axis=1))

    try:
        auc_macro = roc_auc_score(y_binary, y_probabilities, average="macro", multi_class="ovr")
        auc_weighted = roc_auc_score(y_binary, y_probabilities, average="weighted", multi_class="ovr")
    except ValueError:
        auc_macro = np.nan
        auc_weighted = np.nan

    final_metrics = pd.DataFrame([
        ["accuracy", accuracy_score(y_holdout_names, y_pred_names)],
        ["balanced_accuracy", balanced_accuracy_score(y_holdout_names, y_pred_names)],
        ["precision_macro", macro_precision],
        ["recall_macro", macro_recall],
        ["f1_macro", macro_f1],
        ["precision_weighted", weighted_precision],
        ["recall_weighted", weighted_recall],
        ["f1_weighted", weighted_f1],
        ["roc_auc_ovr_macro", auc_macro],
        ["roc_auc_ovr_weighted", auc_weighted],
        ["multiclass_log_loss", log_loss(y_holdout_names, y_probabilities, labels=CLASS_ORDER)],
        ["multiclass_brier_score", brier_multiclass],
        ["cohen_kappa", cohen_kappa_score(y_holdout_names, y_pred_names, labels=CLASS_ORDER)],
        ["matthews_correlation_coefficient", matthews_corrcoef(y_holdout_names, y_pred_names)],
        ["mean_prediction_confidence", float(np.mean(confidence))],
        ["low_confidence_percentage", float(np.mean(confidence < LOW_CONFIDENCE_THRESHOLD) * 100)],
        ["extreme_low_high_error_rate", float(np.mean(error_distance >= 2))],
        ["prediction_seconds", final_prediction_seconds],
    ], columns=["metric", "value"])

    per_class_metrics = pd.DataFrame({
        "class": CLASS_ORDER,
        "precision": class_precision,
        "recall": class_recall,
        "f1_score": class_f1,
        "support": class_support.astype(int),
    })

    report_dictionary = classification_report(
        y_holdout_names,
        y_pred_names,
        labels=CLASS_ORDER,
        output_dict=True,
        zero_division=0,
    )
    classification_report_table = pd.DataFrame(report_dictionary).T.reset_index().rename(columns={"index": "class_or_average"})

    count_matrix = confusion_matrix(y_holdout_names, y_pred_names, labels=CLASS_ORDER)
    normalised_matrix = confusion_matrix(y_holdout_names, y_pred_names, labels=CLASS_ORDER, normalize="true")
    confusion_counts = pd.DataFrame(count_matrix, index=[f"Actual_{c}" for c in CLASS_ORDER], columns=[f"Predicted_{c}" for c in CLASS_ORDER]).reset_index(names="actual_class")
    confusion_normalised = pd.DataFrame(normalised_matrix, index=[f"Actual_{c}" for c in CLASS_ORDER], columns=[f"Predicted_{c}" for c in CLASS_ORDER]).reset_index(names="actual_class")

    save_dataframe(final_metrics, CELL_FOLDERS, "08_06_01_final_metrics")
    save_dataframe(per_class_metrics, CELL_FOLDERS, "08_06_01_per_class_metrics")
    save_dataframe(classification_report_table, CELL_FOLDERS, "08_06_01_classification_report")
    save_dataframe(confusion_counts, CELL_FOLDERS, "08_06_01_confusion_matrix_counts")
    save_dataframe(confusion_normalised, CELL_FOLDERS, "08_06_01_confusion_matrix_normalised")

    display(final_metrics.round(5))
    display(per_class_metrics.round(5))
else:
    print("Skipped: final metrics require FINAL_HOLDOUT mode.")

Skipped: final metrics require FINAL_HOLDOUT mode.


In [10]:
# Cell 8.6.2 — Compare Frozen Cross-Validation Estimates with Final Holdout Results

CELL_FOLDERS = create_cell_folders(
    "08_06", "Comprehensive_Final_Metrics", "08_06_02", "Compare_CV_and_Holdout"
)

cv_holdout_comparison = pd.DataFrame()

if FINAL_MODE:
    finalist_record = next(
        (
            item for item in phase_7_manifest.get("phase_6_finalists", [])
            if item["model"] == primary_candidate["model"] and item["stage"] == primary_candidate["stage"]
        ),
        {},
    )
    holdout_metric_map = dict(final_metrics.values)
    cv_holdout_comparison = pd.DataFrame([
        {
            "metric": "Macro F1",
            "cross_validation_estimate": finalist_record.get("mean_macro_f1", np.nan),
            "final_holdout_result": holdout_metric_map.get("f1_macro", np.nan),
        },
        {
            "metric": "Balanced Accuracy",
            "cross_validation_estimate": finalist_record.get("mean_balanced_accuracy", np.nan),
            "final_holdout_result": holdout_metric_map.get("balanced_accuracy", np.nan),
        },
    ])
    cv_holdout_comparison["holdout_minus_cv"] = (
        cv_holdout_comparison["final_holdout_result"]
        - cv_holdout_comparison["cross_validation_estimate"]
    )
    save_dataframe(cv_holdout_comparison, CELL_FOLDERS, "08_06_02_cv_holdout_comparison")
    display(cv_holdout_comparison.round(5))
else:
    print("Skipped: holdout comparison is unavailable in DRY_RUN mode.")

Skipped: holdout comparison is unavailable in DRY_RUN mode.


# Section 8.7 — Calibration, ROC, and Prediction Confidence

**Purpose:** Describe probability quality and uncertainty without choosing a new threshold or modifying the final model on the holdout.

In [11]:
# Cell 8.7.1 — Calculate ROC and Calibration Data

CELL_FOLDERS = create_cell_folders(
    "08_07", "Calibration_ROC_and_Confidence", "08_07_01", "Calculate_ROC_and_Calibration_Data"
)

roc_curve_data = pd.DataFrame()
calibration_data = pd.DataFrame()
calibration_summary = pd.DataFrame()

if FINAL_MODE:
    y_binary = label_binarize(y_holdout_names, classes=CLASS_ORDER)
    roc_rows = []
    calibration_rows = []
    calibration_summary_rows = []

    for class_index, class_name in enumerate(CLASS_ORDER):
        false_positive_rate, true_positive_rate, thresholds = roc_curve(
            y_binary[:, class_index], y_probabilities[:, class_index]
        )
        class_auc = roc_auc_score(y_binary[:, class_index], y_probabilities[:, class_index])
        for fpr_value, tpr_value, threshold_value in zip(false_positive_rate, true_positive_rate, thresholds):
            roc_rows.append({
                "class": class_name,
                "false_positive_rate": fpr_value,
                "true_positive_rate": tpr_value,
                "threshold": threshold_value,
                "auc": class_auc,
            })

        observed_fraction, predicted_mean = calibration_curve(
            y_binary[:, class_index],
            y_probabilities[:, class_index],
            n_bins=CALIBRATION_BINS,
            strategy="quantile",
        )
        for bin_number, (observed, predicted) in enumerate(zip(observed_fraction, predicted_mean), start=1):
            calibration_rows.append({
                "class": class_name,
                "bin": bin_number,
                "mean_predicted_probability": predicted,
                "observed_fraction": observed,
                "absolute_calibration_error": abs(observed - predicted),
            })
        calibration_summary_rows.append({
            "class": class_name,
            "mean_absolute_calibration_error": float(np.mean(np.abs(observed_fraction - predicted_mean))),
            "maximum_absolute_calibration_error": float(np.max(np.abs(observed_fraction - predicted_mean))),
        })

    roc_curve_data = pd.DataFrame(roc_rows)
    calibration_data = pd.DataFrame(calibration_rows)
    calibration_summary = pd.DataFrame(calibration_summary_rows)

    save_dataframe(roc_curve_data, CELL_FOLDERS, "08_07_01_roc_curve_data")
    save_dataframe(calibration_data, CELL_FOLDERS, "08_07_01_calibration_curve_data")
    save_dataframe(calibration_summary, CELL_FOLDERS, "08_07_01_calibration_summary")
    display(calibration_summary.round(5))
else:
    print("Skipped: probability-quality analysis requires FINAL_HOLDOUT mode.")

Skipped: probability-quality analysis requires FINAL_HOLDOUT mode.


# Section 8.8 — Family-Level Bootstrap Confidence Intervals

**Purpose:** Quantify uncertainty around final holdout metrics by resampling complete families, preserving the study's grouped data structure.

In [12]:
# Cell 8.8.1 — Calculate Family-Level Bootstrap Confidence Intervals

CELL_FOLDERS = create_cell_folders(
    "08_08", "Family_Bootstrap_Confidence_Intervals", "08_08_01", "Calculate_Family_Bootstrap_CI"
)

bootstrap_results = pd.DataFrame()
bootstrap_confidence_intervals = pd.DataFrame()

if FINAL_MODE:
    rng = np.random.default_rng(RANDOM_STATE)
    family_to_indices = {
        family: np.flatnonzero(holdout_groups.to_numpy() == family)
        for family in holdout_groups.unique()
    }
    unique_families = np.array(list(family_to_indices))
    bootstrap_rows = []

    for iteration in range(BOOTSTRAP_ITERATIONS):
        sampled_families = rng.choice(unique_families, size=len(unique_families), replace=True)
        sampled_indices = np.concatenate([family_to_indices[family] for family in sampled_families])
        sampled_true = y_holdout_names.iloc[sampled_indices]
        sampled_pred = y_pred_names[sampled_indices]

        if sampled_true.nunique() < len(CLASS_ORDER):
            continue

        bootstrap_rows.append({
            "iteration": iteration + 1,
            "accuracy": accuracy_score(sampled_true, sampled_pred),
            "balanced_accuracy": balanced_accuracy_score(sampled_true, sampled_pred),
            "precision_macro": precision_score(sampled_true, sampled_pred, labels=CLASS_ORDER, average="macro", zero_division=0),
            "recall_macro": recall_score(sampled_true, sampled_pred, labels=CLASS_ORDER, average="macro", zero_division=0),
            "f1_macro": f1_score(sampled_true, sampled_pred, labels=CLASS_ORDER, average="macro", zero_division=0),
            "f1_weighted": f1_score(sampled_true, sampled_pred, labels=CLASS_ORDER, average="weighted", zero_division=0),
            "recall_low": recall_score(sampled_true, sampled_pred, labels=CLASS_ORDER, average=None, zero_division=0)[0],
            "recall_medium": recall_score(sampled_true, sampled_pred, labels=CLASS_ORDER, average=None, zero_division=0)[1],
            "recall_high": recall_score(sampled_true, sampled_pred, labels=CLASS_ORDER, average=None, zero_division=0)[2],
        })

    bootstrap_results = pd.DataFrame(bootstrap_rows)
    ci_rows = []
    for metric_name in [column for column in bootstrap_results.columns if column != "iteration"]:
        values = bootstrap_results[metric_name].dropna()
        ci_rows.append({
            "metric": metric_name,
            "bootstrap_mean": values.mean(),
            "bootstrap_standard_error": values.std(ddof=1),
            "ci_2_5_percent": values.quantile(0.025),
            "ci_97_5_percent": values.quantile(0.975),
            "valid_iterations": len(values),
        })
    bootstrap_confidence_intervals = pd.DataFrame(ci_rows)

    save_dataframe(bootstrap_results, CELL_FOLDERS, "08_08_01_family_bootstrap_results")
    save_dataframe(bootstrap_confidence_intervals, CELL_FOLDERS, "08_08_01_bootstrap_confidence_intervals")
    display(bootstrap_confidence_intervals.round(5))
else:
    print("Skipped: bootstrap confidence intervals require FINAL_HOLDOUT mode.")

Skipped: bootstrap confidence intervals require FINAL_HOLDOUT mode.


# Section 8.9 — Final Subgroup and Fairness Audit

**Purpose:** Report final performance across available demographic, school, socio-economic, health, and educational subgroups as a diagnostic safeguard, not as a basis for changing the model after holdout access.

In [13]:
# Cell 8.9.1 — Calculate Final Subgroup Performance and Gaps

CELL_FOLDERS = create_cell_folders(
    "08_09", "Final_Subgroup_and_Fairness_Audit", "08_09_01", "Calculate_Subgroup_Performance"
)

subgroup_performance = pd.DataFrame()
subgroup_gaps = pd.DataFrame()

if FINAL_MODE:
    subgroup_variables = [
        variable for variable in [
            "gender",
            "school_type",
            "family_income",
            "stage",
            "chronic_disease",
        ]
        if variable in holdout_data.columns
    ]

    subgroup_rows = []
    gap_rows = []

    for variable in subgroup_variables:
        variable_rows = []
        for group_value, indices in holdout_data.groupby(variable, dropna=False).groups.items():
            indices = np.asarray(list(indices), dtype=int)
            if len(indices) < MINIMUM_SUBGROUP_SIZE:
                continue
            true_group = y_holdout_names.iloc[indices]
            pred_group = y_pred_names[indices]
            recalls = recall_score(true_group, pred_group, labels=CLASS_ORDER, average=None, zero_division=0)
            row = {
                "subgroup_variable": variable,
                "subgroup_value": group_value,
                "record_count": len(indices),
                "accuracy": accuracy_score(true_group, pred_group),
                "balanced_accuracy": balanced_accuracy_score(true_group, pred_group),
                "f1_macro": f1_score(true_group, pred_group, labels=CLASS_ORDER, average="macro", zero_division=0),
                "recall_low": recalls[0],
                "recall_medium": recalls[1],
                "recall_high": recalls[2],
            }
            subgroup_rows.append(row)
            variable_rows.append(row)

        if len(variable_rows) >= 2:
            variable_frame = pd.DataFrame(variable_rows)
            for metric_name in ["accuracy", "balanced_accuracy", "f1_macro", "recall_low", "recall_medium", "recall_high"]:
                gap_rows.append({
                    "subgroup_variable": variable,
                    "metric": metric_name,
                    "maximum_value": variable_frame[metric_name].max(),
                    "minimum_value": variable_frame[metric_name].min(),
                    "absolute_gap": variable_frame[metric_name].max() - variable_frame[metric_name].min(),
                    "groups_evaluated": len(variable_frame),
                })

    subgroup_performance = pd.DataFrame(subgroup_rows)
    subgroup_gaps = pd.DataFrame(gap_rows)
    save_dataframe(subgroup_performance, CELL_FOLDERS, "08_09_01_subgroup_performance")
    save_dataframe(subgroup_gaps, CELL_FOLDERS, "08_09_01_subgroup_gaps")
    display(subgroup_gaps.sort_values("absolute_gap", ascending=False).head(20).round(5))
else:
    print("Skipped: subgroup evaluation requires FINAL_HOLDOUT mode.")

Skipped: subgroup evaluation requires FINAL_HOLDOUT mode.


# Section 8.10 — Final Prediction Pipeline

**Purpose:** Provide a reusable function that accepts new structured records, validates required predictors, returns class probabilities, flags low-confidence cases for human review, and never replaces professional educational judgment.

In [14]:
# Cell 8.10.1 — Define and Test the Reusable Prediction Function

CELL_FOLDERS = create_cell_folders(
    "08_10", "Final_Prediction_Pipeline", "08_10_01", "Define_Reusable_Prediction_Function"
)


def predict_new_students(new_records, model_bundle=FINAL_MODEL_BUNDLE, confidence_threshold=LOW_CONFIDENCE_THRESHOLD):
    """Predict Low/Medium/High with probabilities and a human-review flag."""
    if isinstance(new_records, dict):
        new_records = pd.DataFrame([new_records])
    elif not isinstance(new_records, pd.DataFrame):
        new_records = pd.DataFrame(new_records)

    required_predictors = model_bundle["predictor_columns"]
    missing = sorted(set(required_predictors) - set(new_records.columns))
    if missing:
        raise ValueError("Missing required predictor columns: " + ", ".join(missing))

    encoded = predict_encoded(model_bundle, new_records[required_predictors])
    probabilities = probability_scores(model_bundle, new_records[required_predictors])
    predicted_names = pd.Series(encoded).map(INVERSE_TARGET_MAPPING).to_numpy()
    confidence_values = probabilities.max(axis=1)

    result = pd.DataFrame({
        "predicted_class": predicted_names,
        "prediction_confidence": confidence_values,
        "human_review_recommended": confidence_values < confidence_threshold,
    })
    for class_index, class_name in enumerate(CLASS_ORDER):
        result[f"probability_{class_name.lower()}"] = probabilities[:, class_index]
    return result


pipeline_smoke_test = predict_new_students(X_train.head(5))
if len(pipeline_smoke_test) != 5:
    raise ValueError("Reusable prediction function failed its smoke test.")

save_dataframe(pipeline_smoke_test, CELL_FOLDERS, "08_10_01_prediction_pipeline_smoke_test")
display(pipeline_smoke_test)

,predicted_class,prediction_confidence,human_review_recommended,probability_low,probability_medium,probability_high
0,High,0.562481,False,0.018604,0.418915,0.562481
1,High,0.727020,False,0.007633,0.265347,0.727020
2,Medium,0.540400,True,0.041211,0.540400,0.418389
3,Medium,0.596056,False,0.088243,0.596056,0.315701
4,Medium,0.553678,False,0.073190,0.553678,0.373132


# Section 8.11 — Multiple Visualisation: Final Evaluation

**Purpose:** Create final publication-quality figures only in `FINAL_HOLDOUT` mode; Dry Run leaves the holdout sealed and skips these plots.

In [15]:
# Cell 8.11.1 — Visualise Confusion Matrices and Per-Class Metrics

CELL_FOLDERS = create_cell_folders(
    "08_11", "Multiple_Visualisation_Final_Evaluation", "08_11_01", "Confusion_and_Class_Metrics"
)

if FINAL_MODE:
    count_matrix = confusion_matrix(y_holdout_names, y_pred_names, labels=CLASS_ORDER)
    normalised_matrix = confusion_matrix(y_holdout_names, y_pred_names, labels=CLASS_ORDER, normalize="true")

    for matrix, title, stem, value_format in [
        (count_matrix, "Final Holdout Confusion Matrix", "08_11_01_confusion_counts", "d"),
        (normalised_matrix, "Final Holdout Normalised Confusion Matrix", "08_11_01_confusion_normalised", ".2f"),
    ]:
        figure, axis = plt.subplots(figsize=(7.5, 6.2))
        image = axis.imshow(matrix, cmap="Blues")
        axis.set_xticks(np.arange(len(CLASS_ORDER)))
        axis.set_yticks(np.arange(len(CLASS_ORDER)))
        axis.set_xticklabels(CLASS_ORDER)
        axis.set_yticklabels(CLASS_ORDER)
        axis.set_xlabel("Predicted class")
        axis.set_ylabel("Actual class")
        for row_index in range(matrix.shape[0]):
            for column_index in range(matrix.shape[1]):
                axis.text(
                    column_index,
                    row_index,
                    format(matrix[row_index, column_index], value_format),
                    ha="center",
                    va="center",
                    fontweight="bold",
                    color="white" if matrix[row_index, column_index] > matrix.max() * 0.55 else THEME["charcoal"],
                )
        style_axis(axis, title, "The candidate was locked before holdout access.", grid_axis=None)
        figure.colorbar(image, ax=axis, fraction=0.045, pad=0.04)
        save_figure(figure, CELL_FOLDERS, stem)
        plt.show()

    figure, axis = plt.subplots(figsize=(10.5, 6.2))
    x_positions = np.arange(len(CLASS_ORDER))
    width = 0.24
    axis.bar(x_positions - width, per_class_metrics["precision"], width=width, label="Precision", color=THEME["maroon"])
    axis.bar(x_positions, per_class_metrics["recall"], width=width, label="Recall", color=THEME["gold"])
    axis.bar(x_positions + width, per_class_metrics["f1_score"], width=width, label="F1-score", color=THEME["teal"])
    axis.set_xticks(x_positions)
    axis.set_xticklabels(CLASS_ORDER)
    axis.set_ylim(0, 1)
    axis.set_ylabel("Score")
    style_axis(axis, "Final Per-Class Metrics", "Precision, Recall, and F1-score on the untouched holdout.")
    axis.legend(ncol=3)
    save_figure(figure, CELL_FOLDERS, "08_11_01_per_class_metrics")
    plt.show()
else:
    print("Skipped: final visualisations require FINAL_HOLDOUT mode.")

Skipped: final visualisations require FINAL_HOLDOUT mode.


In [16]:
# Cell 8.11.2 — Visualise ROC, Calibration, Confidence, and CV Comparison

CELL_FOLDERS = create_cell_folders(
    "08_11", "Multiple_Visualisation_Final_Evaluation", "08_11_02", "ROC_Calibration_Confidence_and_CV"
)

if FINAL_MODE:
    figure, axis = plt.subplots(figsize=(8.5, 7))
    for class_name in CLASS_ORDER:
        subset = roc_curve_data[roc_curve_data["class"] == class_name]
        axis.plot(
            subset["false_positive_rate"],
            subset["true_positive_rate"],
            linewidth=2.2,
            color=CLASS_COLORS[class_name],
            label=f"{class_name} (AUC={subset['auc'].iloc[0]:.3f})",
        )
    axis.plot([0, 1], [0, 1], linestyle="--", color="#777777", linewidth=1.2)
    axis.set_xlabel("False Positive Rate")
    axis.set_ylabel("True Positive Rate")
    style_axis(axis, "One-vs-Rest ROC Curves", "Final holdout probability discrimination.", "both")
    axis.legend()
    save_figure(figure, CELL_FOLDERS, "08_11_02_roc_curves")
    plt.show()

    figure, axis = plt.subplots(figsize=(8.5, 7))
    for class_name in CLASS_ORDER:
        subset = calibration_data[calibration_data["class"] == class_name]
        axis.plot(
            subset["mean_predicted_probability"],
            subset["observed_fraction"],
            marker="o",
            linewidth=2,
            color=CLASS_COLORS[class_name],
            label=class_name,
        )
    axis.plot([0, 1], [0, 1], linestyle="--", color="#777777", linewidth=1.2, label="Perfect calibration")
    axis.set_xlabel("Mean predicted probability")
    axis.set_ylabel("Observed fraction")
    style_axis(axis, "Multiclass Calibration Curves", "Descriptive only; no calibration was fitted on the holdout.", "both")
    axis.legend()
    save_figure(figure, CELL_FOLDERS, "08_11_02_calibration_curves")
    plt.show()

    figure, axis = plt.subplots(figsize=(10.5, 6.2))
    correct_confidence = final_predictions.loc[final_predictions["correct_prediction"], "prediction_confidence"]
    incorrect_confidence = final_predictions.loc[~final_predictions["correct_prediction"], "prediction_confidence"]
    axis.hist(correct_confidence, bins=15, alpha=0.65, label="Correct", color=THEME["teal"])
    axis.hist(incorrect_confidence, bins=15, alpha=0.65, label="Incorrect", color=THEME["coral"])
    axis.axvline(LOW_CONFIDENCE_THRESHOLD, linestyle="--", color=THEME["deep_maroon"], label="Human-review threshold")
    axis.set_xlabel("Prediction confidence")
    axis.set_ylabel("Number of records")
    style_axis(axis, "Prediction Confidence by Correctness", "Low-confidence flags support human review; they do not change the predicted class.")
    axis.legend()
    save_figure(figure, CELL_FOLDERS, "08_11_02_confidence_distribution")
    plt.show()

    figure, axis = plt.subplots(figsize=(9.5, 5.8))
    comparison_plot = cv_holdout_comparison.set_index("metric")[["cross_validation_estimate", "final_holdout_result"]]
    comparison_plot.plot(kind="bar", ax=axis, color=[THEME["gold"], THEME["maroon"]])
    axis.set_ylim(0, 1)
    axis.set_ylabel("Score")
    axis.set_xlabel("")
    axis.tick_params(axis="x", rotation=0)
    style_axis(axis, "Cross-Validation versus Final Holdout", "The holdout result is reported, not used to reselect the model.")
    axis.legend(["Cross-validation", "Final holdout"])
    save_figure(figure, CELL_FOLDERS, "08_11_02_cv_vs_holdout")
    plt.show()
else:
    print("Skipped: final visualisations require FINAL_HOLDOUT mode.")

Skipped: final visualisations require FINAL_HOLDOUT mode.


# Section 8.12 — Final Model Card, Quality Checks, and Export

**Purpose:** Export the locked model, predictions, metrics, uncertainty, fairness diagnostics, model card, completion manifest, and complete archive. In Dry Run, export only the locked pipeline and a sealed-holdout status report.

In [17]:
# Cell 8.12.1 — Create the Final Model Card and Research Summary

CELL_FOLDERS = create_cell_folders(
    "08_12", "Final_Model_Card_and_Export", "08_12_01", "Create_Final_Model_Card"
)

holdout_metric_dictionary = dict(final_metrics.values) if FINAL_MODE else {}

model_card = {
    "project_title": PROJECT_TITLE,
    "programme": PROGRAMME,
    "student_name": STUDENT_NAME,
    "student_id": STUDENT_ID,
    "phase": "Phase 8 — Final Model, Locked Holdout Evaluation, and Prediction Pipeline",
    "phase_8_mode": PHASE_8_MODE,
    "evaluation_status": "COMPLETED" if FINAL_MODE else "LOCKED — HOLDOUT SEALED",
    "provisional_until_phase_4_retuned": PROVISIONAL_UNTIL_PHASE_4_RETUNED,
    "locked_model": primary_candidate["model"],
    "locked_stage": primary_candidate["stage"],
    "model_locked_before_holdout": True,
    "training_records": len(training_data),
    "training_families": int(training_groups.nunique()),
    "holdout_records": int(len(holdout_data)) if FINAL_MODE else None,
    "holdout_families": int(holdout_groups.nunique()) if FINAL_MODE else None,
    "predictor_columns": PREDICTOR_COLUMNS,
    "target_classes": CLASS_ORDER,
    "target_mapping": TARGET_MAPPING,
    "final_metrics": holdout_metric_dictionary,
    "top_phase_7_consensus_features": phase_7_manifest.get("top_consensus_features", []),
    "human_review_threshold": LOW_CONFIDENCE_THRESHOLD,
    "intended_use": (
        "Academic research and decision support for understanding patterns associated with student performance."
    ),
    "prohibited_use": [
        "Automatic punishment, exclusion, labelling, or denial of educational opportunities",
        "Replacing professional educational judgment",
        "Using predictions as proof of causation",
        "Selecting a different model or threshold after viewing the final holdout results",
    ],
    "limitations": [
        "Performance depends on the representativeness and quality of the underlying questionnaire or ministry data.",
        "Socio-economic variables may contain measurement and self-reporting error.",
        "Subgroup estimates may be uncertain for small groups.",
        "The present run remains provisional until the planned deeper Phase 4 hyperparameter search is frozen.",
    ],
    "created_at": datetime.now().isoformat(),
}

MODEL_CARD_PATH = save_json(model_card, PHASE_DIR / "phase_08_final_model_card.json")
save_json(model_card, CELL_FOLDERS["reports"] / "08_12_01_final_model_card.json")

summary_lines = [
    "PHASE 8 SUMMARY",
    "=" * 76,
    f"Mode: {PHASE_8_MODE}",
    f"Locked model: {primary_candidate['model']} — {primary_candidate['stage']}",
    f"Training records: {len(training_data):,}",
    f"Final model bundle: {FINAL_MODEL_BUNDLE_PATH}",
    f"Holdout loaded: {FINAL_MODE}",
]
if FINAL_MODE:
    summary_lines.extend([
        f"Holdout records: {len(holdout_data):,}",
        f"Accuracy: {holdout_metric_dictionary['accuracy']:.4f}",
        f"Balanced Accuracy: {holdout_metric_dictionary['balanced_accuracy']:.4f}",
        f"Macro F1: {holdout_metric_dictionary['f1_macro']:.4f}",
        f"Weighted F1: {holdout_metric_dictionary['f1_weighted']:.4f}",
    ])
else:
    summary_lines.append("The final holdout remains sealed pending the expanded Phase 4 tuning.")

SUMMARY_PATH = CELL_FOLDERS["reports"] / "08_12_01_phase_8_summary.txt"
SUMMARY_PATH.write_text("\n".join(summary_lines), encoding="utf-8")
print("\n".join(summary_lines))

PHASE 8 SUMMARY
Mode: DRY_RUN
Locked model: Logistic Regression — Baseline
Training records: 2,006
Final model bundle: C:\Users\User\Desktop\All\outputs\Phase_08_Final_Model_and_Holdout_Evaluation\models\phase_08_final_model_bundle.joblib
Holdout loaded: False
The final holdout remains sealed pending the expanded Phase 4 tuning.


In [18]:
# Cell 8.12.2 — Export Official Phase 8 Tables and Completion Manifest

CELL_FOLDERS = create_cell_folders(
    "08_12", "Final_Model_Card_and_Export", "08_12_02", "Export_Official_Phase_8_Outputs"
)

exported_files = [FINAL_MODEL_BUNDLE_PATH, MODEL_CARD_PATH, CANDIDATE_LOCK_PATH]

if FINAL_MODE:
    official_tables = {
        "phase_08_final_holdout_predictions.csv": final_predictions,
        "phase_08_final_metrics.csv": final_metrics,
        "phase_08_per_class_metrics.csv": per_class_metrics,
        "phase_08_classification_report.csv": classification_report_table,
        "phase_08_confusion_matrix_counts.csv": confusion_counts,
        "phase_08_confusion_matrix_normalised.csv": confusion_normalised,
        "phase_08_cv_holdout_comparison.csv": cv_holdout_comparison,
        "phase_08_bootstrap_confidence_intervals.csv": bootstrap_confidence_intervals,
        "phase_08_subgroup_performance.csv": subgroup_performance,
        "phase_08_subgroup_gaps.csv": subgroup_gaps,
        "phase_08_calibration_summary.csv": calibration_summary,
        "phase_08_roc_curve_data.csv": roc_curve_data,
        "phase_08_calibration_curve_data.csv": calibration_data,
    }
    for filename, dataframe in official_tables.items():
        path = OFFICIAL_DATA_DIR / filename
        dataframe.to_csv(path, index=False, encoding="utf-8-sig")
        exported_files.append(path)

completion_manifest = {
    "completed_phase": "Phase 8 — Final Model and Prediction Pipeline",
    "mode": PHASE_8_MODE,
    "final_evaluation_completed": FINAL_MODE,
    "holdout_used": FINAL_MODE,
    "locked_model": primary_candidate["model"],
    "locked_stage": primary_candidate["stage"],
    "final_model_bundle": str(FINAL_MODEL_BUNDLE_PATH),
    "model_card": str(MODEL_CARD_PATH),
    "candidate_lock": str(CANDIDATE_LOCK_PATH),
    "provisional_until_phase_4_retuned": PROVISIONAL_UNTIL_PHASE_4_RETUNED,
    "official_outputs": [str(path) for path in exported_files],
    "next_phase": "Phase 9 — Open-Ended Response Analysis" if FINAL_MODE else "Return to expanded Phase 4 tuning before final holdout unlock",
    "generated_at": datetime.now().isoformat(),
}

COMPLETION_MANIFEST_PATH = PHASE_DIR / "phase_08_completion_manifest.json"
save_json(completion_manifest, COMPLETION_MANIFEST_PATH)
exported_files.append(COMPLETION_MANIFEST_PATH)

export_summary = pd.DataFrame([
    {
        "artifact": path.name,
        "path": str(path),
        "file_size_kb": round(path.stat().st_size / 1024, 2),
        "sha256": sha256_file(path),
    }
    for path in exported_files
    if Path(path).exists()
])
save_dataframe(export_summary, CELL_FOLDERS, "08_12_02_export_summary")
display(export_summary)

,artifact,path,file_size_kb,sha256
0,phase_08_final_model_bundle.joblib,C:\Users\User\Desktop\All\outputs\Phase_08_Fin...,14.46,f73b644c0c5a8cc05c77e27956becf9ae43974ca0a2f66...
1,phase_08_final_model_card.json,C:\Users\User\Desktop\All\outputs\Phase_08_Fin...,3.97,5c2f4c2656dcbac6508af695cf08ddb79aa8e1225715e4...
2,phase_08_locked_candidate.json,C:\Users\User\Desktop\All\governance\phase_08_...,1.26,f0953c5bf121d348eea60598a666d8e96bb095a980b7f7...
3,phase_08_completion_manifest.json,C:\Users\User\Desktop\All\outputs\Phase_08_Fin...,1.15,efb37191e043b2f10b1c1dc794f9b8623e5ebb30cfe46a...


In [19]:
# Cell 8.12.3 — Run Final Quality Checks and Create the Complete ZIP Archive

CELL_FOLDERS = create_cell_folders(
    "08_12", "Final_Model_Card_and_Export", "08_12_03", "Quality_Checks_and_Complete_Archive"
)

quality_checks = []


def add_check(name, passed, details):
    quality_checks.append({"quality_check": name, "passed": bool(passed), "details": str(details)})


add_check("Candidate locked before holdout", candidate_lock["locked_before_holdout_access"], CANDIDATE_LOCK_PATH)
add_check("Training hash matches Phase 3", actual_training_hash == phase_3_manifest.get("training_dataset_sha256", actual_training_hash), actual_training_hash)
add_check("Final model bundle exists", FINAL_MODEL_BUNDLE_PATH.exists(), FINAL_MODEL_BUNDLE_PATH)
add_check("Prediction pipeline smoke test passed", len(pipeline_smoke_test) == 5, len(pipeline_smoke_test))
add_check("Dry Run keeps holdout sealed", FINAL_MODE or holdout_data is None, PHASE_8_MODE)

if FINAL_MODE:
    add_check("Holdout access log exists", HOLDOUT_ACCESS_LOG_PATH.exists(), HOLDOUT_ACCESS_LOG_PATH)
    add_check("Family overlap is zero", len(set(training_groups.unique()).intersection(set(holdout_groups.unique()))) == 0, "zero required")
    add_check("Prediction exists for every holdout record", len(final_predictions) == len(holdout_data), f"{len(final_predictions)} of {len(holdout_data)}")
    add_check("Probabilities sum to one", np.allclose(y_probabilities.sum(axis=1), 1.0, atol=1e-6), "row-wise probability sum")
    add_check("All principal final metrics are finite", np.isfinite(final_metrics["value"].astype(float)).all(), final_metrics["metric"].tolist())
    add_check("Model was not reselected on holdout", True, "Locked Phase 6 primary candidate used")
else:
    add_check("Final holdout evaluation intentionally incomplete", True, "Awaiting frozen expanded Phase 4 tuning")

quality_checks_table = pd.DataFrame(quality_checks)
save_dataframe(quality_checks_table, CELL_FOLDERS, "08_12_03_phase_8_quality_checks")
save_json(quality_checks, CELL_FOLDERS["reports"] / "08_12_03_phase_8_quality_checks.json")

if not quality_checks_table["passed"].all():
    failed = quality_checks_table.loc[~quality_checks_table["passed"], "quality_check"].tolist()
    raise AssertionError("Phase 8 quality checks failed: " + "; ".join(failed))

ZIP_PATH = PHASE_DIR / "phase_08_complete_export.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(PHASE_DIR.rglob("*")):
        if path.is_file() and path != ZIP_PATH:
            archive.write(path, path.relative_to(PHASE_DIR))

print("All Phase 8 quality checks passed.")
print("Complete archive:", ZIP_PATH)
display(quality_checks_table)

All Phase 8 quality checks passed.
Complete archive: C:\Users\User\Desktop\All\outputs\Phase_08_Final_Model_and_Holdout_Evaluation\phase_08_complete_export.zip


,quality_check,passed,details
0,Candidate locked before holdout,True,C:\Users\User\Desktop\All\governance\phase_08_...
1,Training hash matches Phase 3,True,0efdd6e3754f8844d7eba3a92b70a27f8e5d202c660ea9...
2,Final model bundle exists,True,C:\Users\User\Desktop\All\outputs\Phase_08_Fin...
3,Prediction pipeline smoke test passed,True,5
4,Dry Run keeps holdout sealed,True,DRY_RUN
5,Final holdout evaluation intentionally incomplete,True,Awaiting frozen expanded Phase 4 tuning


# Phase 8 Status

- In **Dry Run**, the final model pipeline is built and tested, but the holdout remains sealed.
- In **Final Holdout**, the candidate is locked first, the holdout is opened once, and all final results are exported.
- Because Phase 4 will be retuned more deeply, do not unlock the final holdout until the final tuning and candidate-selection decisions are frozen.